
# Autonomous Trailer — Knowledge-Based Agent

### ICS 3202: Artificial Intelligence
### Practical Exercise 4
##### 176128 Wachira Christine
##### 190400 Kilwake Abraham
##### 192640 Shalo Natalie

## Introduction

This project develops a knowledge-based agent for an autonomous trailer travelling from Nairobi to Kisumu.

The agent uses a knowledge base containing towns, road distances, and road conditions. It applies logical rules to determine which roads can be used and Uniform Cost Search (UCS) to find the shortest available route.


In [3]:

# STEP 1: Create the Knowledge Base

# Roads and distances from Practical Exercise 3
ROADS = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Narok", 143),
    ("Nairobi", "Nyeri", 151),
    ("Nakuru", "Kericho", 107),
    ("Nakuru", "Nyahururu", 64),
    ("Nyeri", "Nyahururu", 104),
    ("Narok", "Bomet", 81),
    ("Bomet", "Kericho", 76.1),
    ("Bomet", "Kisii", 80),
    ("Kisii", "Kisumu", 115),
    ("Kericho", "Kisumu", 84)
]

# Store road distances and conditions
graph = {}
road_status = {}

for a, b, distance in ROADS:
    # Allow travel in both directions
    graph.setdefault(a, {})[b] = distance
    graph.setdefault(b, {})[a] = distance

    # Initially assume every road is open
    road_status[(a, b)] = "open"
    road_status[(b, a)] = "open"

# Journey information
start = "Nairobi"
goal = "Kisumu"

print("Knowledge Base Created Successfully!")
print("Number of towns:", len(graph))
print("Number of directed road connections:", len(road_status))
print("Starting point:", start)
print("Destination:", goal)


Knowledge Base Created Successfully!
Number of towns: 9
Number of directed road connections: 22
Starting point: Nairobi
Destination: Kisumu


## Step 2: Inference Engine.

In [4]:

# STEP 2: Implement the Inference Engine

def can_travel(source, destination):
    # Rule 1: Check whether the road exists
    if destination not in graph.get(source, {}):
        return False, "No direct road exists."

    # Rule 2: Check whether the road is open
    if road_status[(source, destination)] == "open":
        return True, "Road exists and is open."

    return False, "Road is blocked."


# Function to change road conditions
def change_road_status(source, destination, status):
    if status not in ("open", "blocked"):
        print("Invalid status.")
        return

    if destination not in graph.get(source, {}):
        print("Road does not exist.")
        return

    road_status[(source, destination)] = status
    road_status[(destination, source)] = status

    print(f"Road status updated to: {status}")


# Test 1: Check an open road
print("Test 1:", can_travel("Nakuru", "Kericho"))

# Test 2: Block the road
change_road_status("Nakuru", "Kericho", "blocked")
print("Test 2:", can_travel("Nakuru", "Kericho"))

# Test 3: Reopen the road
change_road_status("Nakuru", "Kericho", "open")
print("Test 3:", can_travel("Nakuru", "Kericho"))


Test 1: (True, 'Road exists and is open.')
Road status updated to: blocked
Test 2: (False, 'Road is blocked.')
Road status updated to: open
Test 3: (True, 'Road exists and is open.')


## Step 3: Route Planning Using Uniform Cost Search.

In [5]:

# STEP 3: Route Planning with UCS

import heapq

def find_best_route(start, goal):
    frontier = [(0, start, [start])]
    visited = {}

    while frontier:
        cost, town, path = heapq.heappop(frontier)

        if town in visited and visited[town] <= cost:
            continue

        visited[town] = cost

        if town == goal:
            return path, cost

        for neighbour, distance in graph.get(town, {}).items():

            # Ask the inference engine whether travel is allowed
            allowed, reason = can_travel(town, neighbour)

            if allowed:
                new_cost = cost + distance
                new_path = path + [neighbour]

                heapq.heappush(
                    frontier,
                    (new_cost, neighbour, new_path)
                )

    return None, float("inf")


# Display the agent's decision
def show_route():
    path, distance = find_best_route(start, goal)

    if path:
        print("Recommended Route:", " -> ".join(path))
        print("Total Distance:", f"{distance:g} km")
    else:
        print("No available route to the destination.")


In [6]:

# Test 1: All roads are open
print("TEST 1: NORMAL ROAD CONDITIONS")
show_route()

print("\nTEST 2: ROAD CLOSURE")

# Simulate a blocked road
change_road_status("Nakuru", "Kericho", "blocked")

# Find an alternative route
show_route()

# Restore the road
change_road_status("Nakuru", "Kericho", "open")


TEST 1: NORMAL ROAD CONDITIONS
Recommended Route: Nairobi -> Nakuru -> Kericho -> Kisumu
Total Distance: 352 km

TEST 2: ROAD CLOSURE
Road status updated to: blocked
Recommended Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total Distance: 384.1 km
Road status updated to: open


## Step 4: Interactive Knowledge-Based Agent

In [8]:

# STEP 4: Interactive Knowledge-Based Agent

def run_agent():
    print("AUTONOMOUS TRAILER AGENT")
    print("Journey: Nairobi to Kisumu")

    while True:
        print("\n1. Find best route")
        print("2. Change road condition")
        print("3. Exit")

        choice = input("Choose an option (1-3): ").strip()

        if choice == "1":
            show_route()

        elif choice == "2":
            source = input("Enter starting town: ").strip().title()
            destination = input("Enter next town: ").strip().title()
            status = input("Enter road status (open/blocked): ").strip().lower()

            if destination not in graph.get(source, {}):
                print("This road does not exist.")
            elif status not in ("open", "blocked"):
                print("Invalid road status.")
            else:
                change_road_status(source, destination, status)
                print("\nUpdated route recommendation:")
                show_route()

        elif choice == "3":
            print("Agent stopped.")
            break

        else:
            print("Invalid option. Try again.")


# Start the agent
run_agent()


AUTONOMOUS TRAILER AGENT
Journey: Nairobi to Kisumu

1. Find best route
2. Change road condition
3. Exit
Recommended Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total Distance: 384.1 km

1. Find best route
2. Change road condition
3. Exit
Recommended Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total Distance: 384.1 km

1. Find best route
2. Change road condition
3. Exit
Road status updated to: open

Updated route recommendation:
Recommended Route: Nairobi -> Nakuru -> Kericho -> Kisumu
Total Distance: 352 km

1. Find best route
2. Change road condition
3. Exit
Agent stopped.


## Testing and Evaluating the Knowledge-Based Agent

In [9]:

# STEP 5: Testing the Knowledge-Based Agent

# Reset all road conditions
def reset_roads():
    for road in road_status:
        road_status[road] = "open"


# Test 1: Normal conditions
reset_roads()
print("TEST 1: ALL ROADS OPEN")
show_route()

# Test 2: One road blocked
print("\nTEST 2: NAKURU-KERICHO ROAD BLOCKED")
change_road_status("Nakuru", "Kericho", "blocked")
show_route()

# Test 3: No route to Kisumu
print("\nTEST 3: NO AVAILABLE ROUTE")
reset_roads()
change_road_status("Kericho", "Kisumu", "blocked")
change_road_status("Kisii", "Kisumu", "blocked")
show_route()

# Test 4: Restore all roads
print("\nTEST 4: ROADS RESTORED")
reset_roads()
show_route()


TEST 1: ALL ROADS OPEN
Recommended Route: Nairobi -> Nakuru -> Kericho -> Kisumu
Total Distance: 352 km

TEST 2: NAKURU-KERICHO ROAD BLOCKED
Road status updated to: blocked
Recommended Route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total Distance: 384.1 km

TEST 3: NO AVAILABLE ROUTE
Road status updated to: blocked
Road status updated to: blocked
No available route to the destination.

TEST 4: ROADS RESTORED
Recommended Route: Nairobi -> Nakuru -> Kericho -> Kisumu
Total Distance: 352 km



## Step 6: Discussion and Evaluation

### Knowledge Representation

The agent uses a knowledge base containing towns, road distances, and road conditions. The road network is represented using a Python dictionary, while road conditions are stored as either "open" or "blocked".

### Logical Reasoning

The inference engine applies a logical rule to determine whether the trailer can use a particular road.

If a road exists and is open, the trailer is allowed to travel on it. Otherwise, the road is excluded from route planning.

### Uniform Cost Search

The agent uses Uniform Cost Search (UCS) to identify the shortest available route from Nairobi to Kisumu.

UCS considers the total distance travelled and selects the route with the lowest cumulative cost while avoiding blocked roads.

### Testing and Observations

The agent was tested under four different conditions:

1. When all roads were open, the shortest route was 352 km.
2. When the Nakuru-Kericho road was blocked, the agent selected an alternative route of 384.1 km.
3. When all roads leading to Kisumu were blocked, the agent reported that no route was available.
4. When the roads were restored, the agent selected the original shortest route.

These results demonstrate that the agent can update its decisions when information in its knowledge base changes.

### Limitations

The road conditions are simulated rather than obtained from real-time traffic information.

The agent also considers distance as the main route cost and does not account for traffic congestion, weather, fuel consumption, or travel time.

### Conclusion

The practical exercise successfully demonstrates a knowledge-based agent that combines knowledge representation, logical inference, and Uniform Cost Search.

The agent uses stored facts and logical rules to make route recommendations and adapt to changes in road conditions.
